# Exp 3 — SPT / LSA Ablation

All runs: vanilla distillation (none), 50 epochs, 15 warmup.

Three runs:
1. **Vanilla + SPT** — Shifted Patch Tokenization only  
   Patch embedding input: 15x 16x 16 (3 channels x  5 copies) -> same N=64 tokens
2. **Vanilla + LSA** — Locality Self-Attention only  
   Diagonal masking + learnable temperature in each attention block
3. **Vanilla + SPT + LSA** — both combined

Compare against Exp 1 (vanilla baseline) to measure contribution of each.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, subprocess, sys

REPO_URL = "YOUR_GIT_REMOTE_HERE"   # TODO: set your git remote URL
REPO_DIR = "/content/repo"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "torchvision", "huggingface-hub",
                    "matplotlib", "seaborn", "scikit-learn"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(".", "..", ".."))

P3_SRC = os.path.join(REPO_DIR, "project3", "src")
P2_SRC = os.path.join(REPO_DIR, "project2", "src")
for p in [P3_SRC, P2_SRC]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(P3_SRC)

In [ ]:
# ── Data + device ────────────────────────────────────────────────────────────
import torch, numpy as np

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/cs148/data/dataset"  # TODO: adjust
else:
    DATA_DIR = os.path.join(REPO_DIR, "project2", "data", "dataset")

BASE_SAVE = "/content/checkpoints" if IN_COLAB else "../checkpoints"

device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("Device:", device)

In [ ]:
# ── Shared config ────────────────────────────────────────────────────────────
import argparse

TEACHER_PATH = os.path.join(REPO_DIR, "project2", "checkpoints", "run10_final_9k", "best_model.pt")

COMMON = dict(
    data_dir=DATA_DIR,
    epochs=100,
    batch_size=128,
    lr=3e-4,
    min_lr=1e-5,
    weight_decay=0.1,
    warmup_epochs=5,
    drop_path_rate=0.2,
    img_size=128,
    repeat_aug=2,
    patience=0,
    num_workers=2,
    seed=42,
    synthetic_n=0,
    val_fraction=0.15,
    distillation="none",
    teacher_path=TEACHER_PATH,
    tau=4.0,
    lambda_dist=0.5,
    resume_checkpoint=None,
)

In [ ]:
# ── Run 3a: Vanilla + SPT ────────────────────────────────────────────────────
import train
torch.manual_seed(42); np.random.seed(42)

args_spt = argparse.Namespace(
    **COMMON,
    save_dir=os.path.join(BASE_SAVE, "exp3a_spt"),
    use_spt=True,
    use_lsa=False,
)
train.train(args_spt)

In [ ]:
# ── Run 3b: Vanilla + LSA ────────────────────────────────────────────────────
import importlib; importlib.reload(train)
torch.manual_seed(42); np.random.seed(42)

args_lsa = argparse.Namespace(
    **COMMON,
    save_dir=os.path.join(BASE_SAVE, "exp3b_lsa"),
    use_spt=False,
    use_lsa=True,
)
train.train(args_lsa)

In [ ]:
# ── Run 3c: Vanilla + SPT + LSA ─────────────────────────────────────────────
importlib.reload(train)
torch.manual_seed(42); np.random.seed(42)

args_spt_lsa = argparse.Namespace(
    **COMMON,
    save_dir=os.path.join(BASE_SAVE, "exp3c_spt_lsa"),
    use_spt=True,
    use_lsa=True,
)
train.train(args_spt_lsa)

In [ ]:
# ── Results summary ──────────────────────────────────────────────────────────
import json

for name, save_dir in [
    ("spt only",  os.path.join(BASE_SAVE, "exp3a_spt")),
    ("lsa only",  os.path.join(BASE_SAVE, "exp3b_lsa")),
    ("spt + lsa", os.path.join(BASE_SAVE, "exp3c_spt_lsa")),
]:
    log_path = os.path.join(save_dir, "training_log.json")
    if os.path.exists(log_path):
        with open(log_path) as f:
            history = json.load(f)
        best_va = max(e["val"]["acc"] for e in history)
        print(f"{name:15s}: best val_acc = {best_va:.4f}")

In [ ]:
# ── Download results (Colab only) ────────────────────────────────────────────
if IN_COLAB:
    import shutil
    from google.colab import files
    for name, save_dir in [
        ("exp3a_spt", os.path.join(BASE_SAVE, "exp3a_spt")),
        ("exp3b_lsa", os.path.join(BASE_SAVE, "exp3b_lsa")),
        ("exp3c_spt_lsa", os.path.join(BASE_SAVE, "exp3c_spt_lsa")),
    ]:
        zip_path = f"/content/{name}_results.zip"
        shutil.make_archive(f"/content/{name}_results", "zip", save_dir)
        files.download(zip_path)